# OmniVoice Project Studio — Google Colab (AI-native)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binhminhanh1235/OmniVoice/blob/master/notebooks/OmniVoice_Project_Studio_Colab.ipynb)

Advanced notebook: Gradio Web UI + REST API + SSE jobs + MCP from one OmniVoice Studio server.

If you only want the temporary Gradio UI with the fewest setup steps, use `OmniVoice_Project_Studio_Colab_Gradio.ipynb`.


In [ ]:
# Persistent startup cache + exact OmniVoice wheel.
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

CACHE_VERSION = "v1"
MODEL_ID = "k2-fsa/OmniVoice"
ASR_MODEL = "openai/whisper-small.en"
WORKSPACE = "/content/OmniVoiceStudio"
PERSISTENT_WORKSPACE = "/content/drive/MyDrive/OmniVoiceStudio"
LOCAL_CACHE_BASE = Path("/content/.cache/omnivoice")
PERSISTENT_CACHE_BASE = Path(PERSISTENT_WORKSPACE) / ".startup-cache"

compatibility = {
    "schema_version": 1,
    "cache_version": CACHE_VERSION,
    "python_version": f"{sys.version_info.major}.{sys.version_info.minor}",
    "system": platform.system().lower(),
    "machine": platform.machine().lower(),
}
cache_payload = json.dumps(compatibility, sort_keys=True, separators=(",", ":"))
CACHE_KEY = f"{CACHE_VERSION}-" + hashlib.sha256(cache_payload.encode()).hexdigest()[:16]
LOCAL_CACHE = LOCAL_CACHE_BASE / CACHE_KEY
PERSISTENT_CACHE = PERSISTENT_CACHE_BASE / CACHE_KEY
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

def _read_metadata(path):
    try:
        value = json.loads((path / "metadata.json").read_text(encoding="utf-8"))
        return value if isinstance(value, dict) else {}
    except Exception:
        return {}

persistent_metadata = _read_metadata(PERSISTENT_CACHE)
RESOURCE_FAST_PATH = persistent_metadata.get("compatibility") == compatibility

cached_ref = persistent_metadata.get("last_package_ref")
try:
    with urllib.request.urlopen(
        "https://api.github.com/repos/binhminhanh1235/OmniVoice/branches/master",
        timeout=15,
    ) as response:
        PACKAGE_REF = json.load(response)["commit"]["sha"]
except Exception as exc:
    if cached_ref:
        PACKAGE_REF = str(cached_ref)
        print("GitHub revision lookup unavailable; using cached revision:", PACKAGE_REF)
    else:
        PACKAGE_REF = "master"
        print("GitHub revision lookup unavailable; cold install will resolve master:", type(exc).__name__)

def _copy_tree(source, destination):
    source = Path(source)
    destination = Path(destination)
    if source.exists():
        shutil.copytree(
            source,
            destination,
            dirs_exist_ok=True,
            symlinks=False,
            ignore=shutil.ignore_patterns("*.tmp", ".nfs*"),
        )

# Restore only bootstrap-critical caches before OmniVoice itself is installed.
if RESOURCE_FAST_PATH:
    _copy_tree(PERSISTENT_CACHE / "pip", LOCAL_CACHE / "pip")
    _copy_tree(PERSISTENT_CACHE / "wheels", LOCAL_CACHE / "wheels")

os.environ["PIP_CACHE_DIR"] = str(LOCAL_CACHE / "pip")
package_key = hashlib.sha256(PACKAGE_REF.encode()).hexdigest()[:16]
wheel_dir = LOCAL_CACHE / "wheels" / package_key
wheel_dir.mkdir(parents=True, exist_ok=True)
wheels = sorted(wheel_dir.glob("omnivoice-*.whl"))

if wheels:
    WHEEL_FAST_PATH = True
    wheel = wheels[-1]
else:
    WHEEL_FAST_PATH = False
    subprocess.check_call([
        sys.executable, "-m", "pip", "wheel", "-q", "--no-deps",
        f"git+https://github.com/binhminhanh1235/OmniVoice.git@{PACKAGE_REF}",
        "--wheel-dir", str(wheel_dir),
    ])
    wheels = sorted(wheel_dir.glob("omnivoice-*.whl"))
    if not wheels:
        raise RuntimeError("OmniVoice wheel build completed without producing a wheel.")
    wheel = wheels[-1]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(wheel)
])

# Restore model/ASR caches only after the cache library is available.
os.environ["OMNIVOICE_LOCAL_CACHE_ROOT"] = str(LOCAL_CACHE_BASE)
os.environ["OMNIVOICE_CACHE_SOURCE"] = str(PERSISTENT_CACHE_BASE)
os.environ["OMNIVOICE_CACHE_PERSIST_ROOT"] = str(PERSISTENT_CACHE_BASE)

from omnivoice.runtime_cache import (
    RuntimeCacheFingerprint,
    apply_cache_environment,
    detect_runtime_cache,
    persist_runtime_cache,
    prepare_runtime_cache,
    write_workspace_cache_metadata,
)

CACHE_FINGERPRINT = RuntimeCacheFingerprint.current(
    cache_version=CACHE_VERSION,
    package_ref=PACKAGE_REF,
)
CACHE_PREPARATION = prepare_runtime_cache(
    detect_runtime_cache(),
    CACHE_FINGERPRINT,
)
apply_cache_environment(CACHE_PREPARATION)

# Warm unchanged model resources now, while the persistent cache boundary is explicit.
from huggingface_hub import snapshot_download
snapshot_download(MODEL_ID)
snapshot_download(ASR_MODEL)
persist_runtime_cache(CACHE_PREPARATION)

Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
write_workspace_cache_metadata(WORKSPACE, CACHE_PREPARATION)

import torch
from omnivoice.hardware_quality import detect_hardware

print("Package revision:", PACKAGE_REF)
print("Resource cache:", "FAST" if CACHE_PREPARATION.fast_path else "COLD")
print("Exact wheel:", "FAST" if WHEEL_FAST_PATH else "BUILT")
print("Local cache:", CACHE_PREPARATION.local_namespace)
print("CUDA:", torch.cuda.is_available())
hardware = detect_hardware()
print(hardware.summary())
for note in hardware.notes:
    print("-", note)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime: Runtime → Change runtime type → T4 GPU")


## Local-first workspace + persistent Drive mirror

Generation runs from `/content/OmniVoiceStudio` on Colab local SSD. Google Drive
stores the persistent copy and the startup cache, but it is not in the render hot
path.

At startup the project workspace is restored once from
`MyDrive/OmniVoiceStudio`. While Studio is running, changed project/checkpoint
files are mirrored back every 45 seconds, followed by a final sync on exit.
The `.startup-cache` directory is excluded from workspace mirroring and is
managed independently by the versioned cache layer.


In [ ]:
import atexit
import threading

SYNC_INTERVAL_SECONDS = 45
Path(PERSISTENT_WORKSPACE).mkdir(parents=True, exist_ok=True)

def _sync_workspace_tree(source, destination, *, delete=False):
    source_path = Path(source)
    destination_path = Path(destination)
    source_path.mkdir(parents=True, exist_ok=True)
    destination_path.mkdir(parents=True, exist_ok=True)
    rsync = shutil.which("rsync")
    if rsync:
        command = [
            rsync, "-a",
            "--exclude=.startup-cache/",
            "--exclude=*.tmp",
            "--exclude=.nfs*",
        ]
        if delete:
            command.append("--delete")
        command.extend([f"{source_path}/", f"{destination_path}/"])
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL)
        return
    shutil.copytree(
        source_path,
        destination_path,
        dirs_exist_ok=True,
        symlinks=False,
        ignore=shutil.ignore_patterns(".startup-cache", "*.tmp", ".nfs*"),
    )

if not globals().get("_OMNIVOICE_LOCAL_RESTORED", False):
    _sync_workspace_tree(PERSISTENT_WORKSPACE, WORKSPACE, delete=False)
    _OMNIVOICE_LOCAL_RESTORED = True
    print("Restored persistent Studio data to local SSD.")

old_stop = globals().get("_OMNIVOICE_SYNC_STOP")
if old_stop is not None:
    old_stop.set()

_OMNIVOICE_SYNC_STOP = threading.Event()

def sync_workspace_to_drive():
    _sync_workspace_tree(WORKSPACE, PERSISTENT_WORKSPACE, delete=True)

def _mirror_loop():
    while not _OMNIVOICE_SYNC_STOP.wait(SYNC_INTERVAL_SECONDS):
        try:
            sync_workspace_to_drive()
        except Exception as exc:
            print("Workspace mirror warning:", type(exc).__name__, exc)

_OMNIVOICE_SYNC_THREAD = threading.Thread(
    target=_mirror_loop,
    name="omnivoice-drive-mirror",
    daemon=True,
)
_OMNIVOICE_SYNC_THREAD.start()
atexit.register(sync_workspace_to_drive)

write_workspace_cache_metadata(WORKSPACE, CACHE_PREPARATION)
print("Execution workspace:", WORKSPACE)
print("Persistent mirror:", PERSISTENT_WORKSPACE)
print(f"Mirror interval: {SYNC_INTERVAL_SECONDS}s")


## Optional stable hostname + private access

Set `USE_STABLE_TUNNEL = True` only after creating a remotely-managed Cloudflare Tunnel pointing your hostname to `http://localhost:8000`.

Create these Colab Secrets:

- `CLOUDFLARE_TUNNEL_TOKEN`
- `OMNIVOICE_API_TOKEN`
- `OMNIVOICE_UI_USERNAME`
- `OMNIVOICE_UI_PASSWORD`

When enabled, the same hostname exposes `/ui`, `/api/v1`, `/mcp`, and `/health`.


In [ ]:
PUBLIC_URL = "https://omnivoice.example.com"
USE_STABLE_TUNNEL = False

import os

if USE_STABLE_TUNNEL:
    from google.colab import userdata

    required_names = [
        "CLOUDFLARE_TUNNEL_TOKEN",
        "OMNIVOICE_API_TOKEN",
        "OMNIVOICE_UI_USERNAME",
        "OMNIVOICE_UI_PASSWORD",
    ]
    required = {}
    for name in required_names:
        try:
            required[name] = userdata.get(name)
        except Exception:
            required[name] = None
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))
    for name, value in required.items():
        os.environ[name] = value
    os.environ["OMNIVOICE_API_TOKEN_SCOPES"] = (
        "omnivoice:read,omnivoice:generate,omnivoice:queue,omnivoice:mcp"
    )
    os.environ["OMNIVOICE_PUBLIC_URL"] = PUBLIC_URL
    del required
    print("Stable private public URL configured:", PUBLIC_URL)


In [ ]:
if USE_STABLE_TUNNEL:
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod 700 /content/cloudflared
    !/content/cloudflared --version


## Launch

- Stable mode: unified server with Gradio `/ui`, REST, SSE, and MCP.
- Simple fallback: temporary Gradio share URL only.


In [ ]:
try:
    if USE_STABLE_TUNNEL:
        !omnivoice-studio serve \
          --model k2-fsa/OmniVoice \
          --workspace "$WORKSPACE" \
          --asr-model openai/whisper-small.en \
          --asr-device cpu \
          --host 0.0.0.0 \
          --port 8000 \
          --tunnel \
          --cloudflared /content/cloudflared \
          --public-url "$OMNIVOICE_PUBLIC_URL"
    else:
        !omnivoice-project-studio \
          --model k2-fsa/OmniVoice \
          --workspace "$WORKSPACE" \
          --asr-model openai/whisper-small.en \
          --asr-device cpu \
          --share
finally:
    _OMNIVOICE_SYNC_STOP.set()
    try:
        sync_workspace_to_drive()
        print("Final Studio workspace sync completed.")
    finally:
        persist_runtime_cache(CACHE_PREPARATION)
        print("Startup/model cache persisted.")
